# AI 최적화 & 엔지니어링

## 0. AI 최적화와 정렬
- 이미 만들어진 AI를 효율적으로 활용하려면? -> 전이학습, 경량화
- AI가 사람의 의도와 가치에 맞게 행동하려면? -> AI 정렬(Alignment)
- 준비된 AI에게 일을 맡기려면? -> 프롬프트 엔지니어링, 에이전트 하네스 & 루프


## 1. 전이학습과 딥러닝 경량화
### 1-1. 전이학습이란?
- 딥러닝 모델의 한계
    - 데이터 부족
    - 과적합 가능성 높음
    - 처음부터 학습하면 시간과 비용이 많이 듦
- 전이학습 : 모델이 사전 학습으로 배운 능력을 활용해 새로운 작업을 학습하는 것
    1. 사전 학습 - 특징을 찾는 방법이 가중치에 담김
    2. 기존 모델의 특징 파악 능력은 그대로 쓰고, 분류층만 새 분류층으로 교체
    3. 정답 레이블이 있는 새 데이터로 새 분류층 학습
    4. 필요시 fine-tuning

    <img src="https://d2l.ai/_images/finetune.svg" width="400" style="background-color:white; padding:10px;">

- 핵심 용어
    - Source : 기존에 학습한 문제
    - Target : 새롭게 해결할 문제
    - Domain : 사용한 데이터의 특성
    - Task : 수행하려는 작업
- Source와 Target의 Domain이나 Task 중 적어도 하나가 다를 때, Source에서 배운 지식을 Target에 활용

### 1-2. 전이학습의 분류
- Domain과 Task에 따른 분류

| 종류 | Domain | Task | 정답 레이블 |
|:---:|:---:|:---:|:---:|
| 귀납적(Inductive) | 같거나 다름 | 다름 | Target : O |
| 변환적(Transductive) | 다름 | 같음 | Source : O, Target : X |
| 비지도(Unsupervised) | 같거나 다름 | 둘 다 비지도 | 둘 다 X |

- 전이 대상에 따른 분류
    - 인스턴스 기반 : 기존 데이터 일부
    - 특성 기반 : 데이터를 나타내는 특징 표현
    - 파라미터 기반 : 기존 모델의 학습된 가중치, 편향
    - 관계 기반 : 데이터나 개체 사이의 관계 패턴

### 1-3. Pre-training & Fine-tuning
- 전통적인 전이학습
    - 이미지 : ImageNet으로 사전 학습한 CNN의 가중치 재사용
    - 언어 : 정적 단어 임베딩 재사용 -> 같은 단어는 항상 같은 벡터라 문맥에 따른 의미 차이 반영 X
- 트랜스포머 이후의 전이학습
    - 이미지 : 자기지도학습(가려진 부분 복원), 이미지·텍스트 공동 학습 -> 정답 레이블 없이도 사전 학습 가능
    - 언어 : BERT(가려진 단어 예측), GPT(다음 토큰 예측) -> 문맥을 반영하는 표현
- Foundation Model : 여러 작업에 활용할 수 있도록 방대한 데이터로 사전 학습한 기반 모델
- Fine-tuning : 사전 학습된 모델을 새 작업의 데이터로 추가 학습하고, 가중치 일부 또는 전체를 조정
    - 특징 추출 -> 예측 -> 오차 계산 -> 가중치 업데이트
- Fine-tuning 방법
    - Full Fine-tuning
        - 모든 가중치 업데이트
        - 메모리, 연산 비용 큼
        - 데이터 부족하면 과적합, Catastrophic Forgetting 발생 가능
    - Partial Fine-tuning
        - 일부 층은 고정, 나머지 층만 업데이트
        - 보통 앞쪽 층(단순한 특징) 고정, 뒤쪽 층(복잡한 특징) 학습
    - Frozen Backbone
        - 기존 모델 전체 고정, 새 분류층만 학습
        - 학습 부담 가장 작음, but 새 도메인 적응에 한계
- 선택 기준 (적응 범위 & 비용 : Frozen Backbone < Partial < Full)
    - 데이터 적음 + 도메인 유사 -> Frozen Backbone
    - 데이터 적음 + 도메인 다름 -> Partial
    - 데이터 많음 + 도메인 유사 -> Partial
    - 데이터 많음 + 도메인 다름 -> Full

### 1-4. PEFT
- PEFT(Parameter-Efficient Fine-Tuning) : 사전 학습 모델의 가중치 대부분을 고정하고, 최소한의 파라미터만 학습하는 방법
- 메모리 부담 감소, Foundation Model은 한 번만 저장하고 작업별 파라미터 묶음만 따로 저장
- LoRA(Low-Rank Adaptation)
    - 기존 가중치 $W$는 고정, 수정 행렬 $\Delta W$를 작은 두 행렬 $A$, $B$의 곱으로 학습
    - $W_{new} = W + \Delta W = W + BA$

    <img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_diagram.png" width="400" style="background-color:white; padding:10px;">

    - ex. 1000 x 1000 가중치 -> 전체 학습 1,000,000개 vs. LoRA(r=8) 16,000개
    - r 작음 -> 비용 감소, 표현에 한계
    - r 큼 -> 다양한 표현 가능, 비용 증가
- Adapter : 기존 모델 중간에 작은 신경망을 넣고 그 가중치만 학습
- Prompt tuning : 모델 입력에 학습 가능한 숫자 벡터를 붙이는 방식
- Zero-shot & Few-shot : 추가 학습 없이 입력만 바꿔서 원하는 결과를 이끌어냄

### 1-5. 전이학습의 한계
- Negative Transfer
    - 전이학습 적용 후 새 작업의 성능이 오히려 낮아지는 경우
    - MMD : Source와 Target 데이터의 분포가 얼마나 다른지 측정 -> 작으면 유사, 크면 다름
- Catastrophic Forgetting
    - 새 데이터를 학습하면서 기존 능력이 약해지는 경우
    - EWC : 기존 작업에서 중요했던 가중치가 크게 변할수록 패널티를 줘서 보존
- Bias Transfer
    - 사전 학습에서 얻은 편향이 새 작업에도 이어지는 경우
- Transferability 측정의 어려움
    - 어떤 모델이 새 작업에 잘 맞을지 미리 알기 어려움
    - LEEP : 기존 모델의 예측 확률 이용, 0에 가까울수록 좋음
    - LogME : 기존 모델이 추출한 특징 벡터 이용, 높을수록 좋음

### 1-6. 딥러닝 모델 경량화
- 학습이 끝난 모델을 사용할 때 드는 메모리와 연산 비용을 줄이는 방법
- 가지치기(Pruning) : 예측에 미치는 영향이 작은 부분을 제거
    - 비구조적 가지치기 / 구조적 가지치기
- 양자화(Quantization) : 숫자를 덜 정밀하게 표현해서 저장 공간을 줄임
    - ex. 32비트 실수 -> 8비트 정수
- 지식 증류(Knowledge Distillation) : 작은 Student 모델이 정답과 Teacher 모델의 출력 분포를 함께 학습
- 모델 구조 경량화 : 설계할 때부터 더 적은 계산으로 수행할 수 있는 구조를 선택
- 선택 기준
    - 재학습이 어려움 -> 학습 후 양자화
    - 재학습 가능 -> 가지치기 후 추가 학습, 지식 증류, 모델 구조 경량화
    - 메모리 부족 -> 양자화
    - 실행 속도 느림 -> 구조적 가지치기, 모델 구조 경량화


## 2. AI Alignment
### 2-1. AI Alignment란?
- 기존 한계 : LLM은 다음 토큰 예측만 배움 -> 사실과 다른 문장, 편향 등 사용자 의도와 다르게 흘러갈 수 있음
- AI Alignment : 어떤 답변이 사람에게 바람직한지 기준을 정하고, 모델이 그 기준에 맞게 답하도록 조정하는 과정
- 정렬 기준
    - Helpful : 사용자의 의도를 파악하고 실질적으로 도움이 되는 답변
    - Honest : 사실에 근거해 답하고, 불확실한 정보는 불확실성을 드러냄
    - Harmless : 사람에게 해를 끼칠 가능성을 줄임
- 두 가지 관점
    - Outer Alignment : 목표를 제대로 설정했는가?
        - 보상 해킹(Reward Hacking) : 보상 기준의 허점을 이용해 점수만 높이는 현상
    - Inner Alignment : 모델이 실제로 무엇을 배웠는가?
        - 기만적 정렬(Deceptive Alignment) : 훈련이나 감독 중에만 사람의 의도에 맞게 행동할 수 있다는 가설적 가능성
        - 해석 가능성(Interpretability) 연구가 중요
- 정렬을 어렵게 만드는 요인
    - 가치 정렬의 모호성 : 사람마다 가치관이 달라서 모두가 동의하는 기준을 만들기 어려움
    - 확장 가능한 감독 : 사람이 직접 확인할 수 있는 범위에 한계

### 2-2. RLHF
- RLHF(Reinforcement Learning from Human Feedback) : 사람의 피드백을 학습에 사용해서 사람들이 선호하는 답변에 높은 점수를 주는 방식
- InstructGPT식 파이프라인 : SFT -> Reward Model -> RL
    1. SFT(Supervised Fine-Tuning) - 사람이 작성한 모범 답변으로 언어 모델 학습
    2. Reward Model - 사람이 선호한 답변에 더 높은 점수를 주는 보상 모델 학습
    3. RL - Reward Model이 채점, 높은 보상을 받는 답변을 더 자주 생성하도록 조정 (PPO, KL 패널티 사용)

    <img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/rlhf/rlhf.png" width="450" style="background-color:white; padding:10px;">

- 부작용
    - Reward Hacking : 좋은 답변보다 높은 보상 점수를 받도록 학습
    - Sycophancy(아첨) : 사실이나 근거보다 사용자 생각에 무조건 동의 -> Honest 위반


## 3. 프롬프트 엔지니어링
### 3-1. 프롬프트 엔지니어링이란?
- AI가 더 좋은 답을 내도록 입력(프롬프트)을 설계하는 기술
- Fine-tuning은 모델을 바꾸고, 프롬프트 엔지니어링은 모델은 그대로 두고 입력만 바꿈

| 구분 | 프롬프트 엔지니어링 | Fine-tuning |
|:---:|:---:|:---:|
| 비용 | 낮음 | 높음 |
| 속도 | 바로 시도, 바로 수정 | 학습 시간 필요 |
| 성능 한계 | 모델이 원래 가진 능력 안에서만 | 모델 자체를 바꿔 특정 도메인에 맞춤 |

- 보통은 프롬프트부터 시도 -> 안 되면 Fine-tuning 고려

### 3-2. 주요 기법
- Zero-shot : 예시 없이 지시만 줌 -> 간단하고 명확한 작업
- Few-shot : 예시를 몇 개 보여줌 -> 출력 형식이나 판단 기준을 맞추고 싶을 때
    - 개수보다 답의 종류를 골고루 보여주는 게 중요
- Chain-of-Thought(CoT) : 중간 풀이 과정을 단계별로 쓰게 함 -> 여러 단계가 필요한 문제
- Role Prompting : 역할을 부여 -> 관점, 말투, 설명 수준 조절
    - 모델이 아는 내용 자체는 변하지 않음
- Instruction Prompting : 형식, 길이, 대상, 제약을 구체적으로 지시 -> 거의 모든 경우의 기본
- Self-Consistency : 같은 문제를 여러 번 풀게 하고 다수결 -> 정확도가 중요할 때, 비용도 횟수만큼 증가
- Instruction Prompting에서 시작해서 필요에 따라 조합

### 3-3. 프롬프트 엔지니어링의 한계
- Prompt Injection
    - 사용자 입력에 지시처럼 보이는 문장을 끼워 넣어서 모델이 개발자의 지시 대신 그 문장을 따르게 만드는 것
    - 모델은 시스템 프롬프트와 사용자 입력을 한 덩어리로 받아서 경계가 확실하지 않음
    - 완화 : 구분자로 분리, 모델 권한 최소화, 출력 검증
- Hallucination
    - 사실이 아닌 내용을 그럴듯하게 생성
    - 완화 : 근거 자료 제공(RAG), "모르면 모른다고 답해줘" 지시, 출처 요구 후 직접 확인
- 모델 의존성
    - 같은 프롬프트도 모델이나 버전이 바뀌면 결과가 달라짐
    - 완화 : 모델 버전 고정, 회귀 테스트
- 성능 한계
    - 모델이 원래 못하는 일은 프롬프트로 해결 X
    - 완화 : 외부 도구 연결, Fine-tuning


## 4. 하네스 엔지니어링
### 4-1. 하네스란?
- 프롬프트는 입력 한 번 -> 출력 한 번, 실제 일은 그렇게 끝나지 않는 경우가 많음
- 에이전트(Agent) : 도구를 쓰며 여러 단계를 스스로 진행하는 AI
- 하네스(Harness) : 모델이 일하도록 둘러싼 시스템

    <img src="https://lilianweng.github.io/posts/2023-06-23-agent/agent-overview.png" width="450" style="background-color:white; padding:10px;">

- 모델이 직접 할 수 있는 건 글을 쓰는 것뿐 -> 도구 실행은 하네스가 맡음
- 모델 자체를 더 똑똑하게 만들어 주지는 않음
- 필요한 경우
    - 모델이 모르는 정보를 가져와야 할 때
    - 실제로 무언가를 실행해야 할 때
    - 결과를 확인하고 다시 시도해야 할 때
    - 위험한 행동을 막아야 할 때

### 4-2. 하네스의 구성 요소
- 지시/컨텍스트 : 모델이 무엇을 보고 시작하나
- 도구 인터페이스 : 모델이 무엇을, 어떤 형식으로 할 수 있나
- 실행 환경 : 행동이 실제로 어디서 실행되나 (샌드박스)
- 상태/메모리 : 지금까지 한 일을 어떻게 기억하나
- 관측/피드백 : 행동의 결과를 어떻게 돌려받나
- 권한/가드레일 : 무엇은 하면 안 되나
- 도구 스키마(Tool Schema) : 모델이 읽는 도구 사용 설명서
    - 언제, 무엇을, 어떤 값으로 호출하는지 명확하게 정의해야 함
- 반환 계약 : 도구가 결과(성공, 실패)를 어떤 형태로 돌려줄지 미리 정한 약속
    - 결과 없음과 조회 실패를 구분해서 돌려줘야 모델이 다음 행동을 정할 수 있음

### 4-3. 실패는 어디서 오는가
- 에이전트가 실패했다고 모델의 추론력만 의심하면 안 됨 -> 인터페이스에 따라서도 성능이 크게 달라짐
- SWE-agent : 모델은 그대로 두고 ACI(Agent-Computer Interface)만 새로 설계 -> 11.0% -> 18.0%
- 증상 -> 수정할 곳
    - 엉뚱한 도구를 부른다 -> 도구 인터페이스
    - 같은 실수를 반복한다 -> 관측/피드백
    - 앞부분을 잊는다 -> 지시/컨텍스트, 상태/메모리
    - 위험한 행동을 바로 실행한다 -> 권한/가드레일, 실행 환경
    - 결과가 맞는지 모른다 -> 관측/피드백


## 5. 루프 엔지니어링
### 5-1. 루프란 무엇인가
- 루프(Loop) : 사람이 하던 반복을 시스템이 하도록 만든 것
- 모델이 행동을 제안 -> 시스템이 실행 -> 결과를 다시 모델에게 돌려주는 과정을 종료 조건까지 반복
- 루프 한 바퀴
    1. 관측 - 지금 상황을 봄
    2. 판단 - 다음 행동을 정함
    3. 행동 - 도구를 실행함
    4. 검증 - 결과가 목표에 맞는지 확인함
- 검증 결과는 다음 바퀴의 관측이 됨
- 하네스는 공간의 설계, 루프는 시간의 설계
- ReAct(Reasoning + Acting)
    - Thought(추론) -> Action(행동) -> Observation(관측)을 번갈아 함
    - 생각만 하면 Hallucination, 행동만 하면 같은 행동 반복 -> 둘을 번갈아 해서 완화

    <img src="https://lilianweng.github.io/posts/2023-06-23-agent/react.png" width="500" style="background-color:white; padding:10px;">

### 5-2. 루프의 구성 요소
- 목표 : 무엇을 끝내려 하나
- 트리거 : 루프가 언제 시작되나
- 관측과 행동 : 매 반복에서 무엇을 보고 무엇을 하나
- 정상 종료 조건 : 성공해서 멈추는 조건
- 안전 상한 : 위험해서 멈추는 조건 (반복 횟수, 시간, 비용)
- 실패/승인 처리 : 막혔을 때, 위험할 때 어떻게 하나
- 정상 종료 조건이 영원히 참이 안 될 수도 있음 -> 안전 상한이 따로 필요
- 모델이 끝났다고 해도 실제로 끝났는지는 따로 검증해야 함
- 루프 패턴 : 재시도, 주기 실행, 이벤트 트리거, 목표 달성까지 반복
    - 어떤 패턴이든 종료 조건과 안전 상한을 같이 적음

### 5-3. 통제
- 루프에서 증폭되는 실패
    - 무한 반복
    - 토큰/비용 폭주
    - 컨텍스트 누적에 따른 품질 저하
    - 같은 부작용의 반복
    - 되돌리기 어려운 행동
- 통제 장치

| 통제 장치 | 막는 실패 |
|:---:|:---:|
| 반복, 시간, 비용 상한 | 무한 반복, 비용 폭주 |
| 테스트와 완료 조건 | 끝나지 않았는데 끝났다고 하는 것 |
| 중복 실행 방지 | 같은 부작용의 반복 |
| 체크포인트와 롤백 | 잘못된 변경이 쌓이는 것 |
| 위험도에 따른 행동 분리 | 되돌리기 어려운 행동의 자동 실행 |

- 위험도에 따른 행동 분리 : 자동 실행 / 사후 검토 / 사람 승인 후 실행
- 학습 단계의 정렬 vs. 실행 단계의 통제
    - 학습 단계의 정렬 : 모델 안(파라미터)을 바꿈 -> RLHF
    - 실행 단계의 통제 : 사람 승인, 권한 최소화, 테스트로 검증
    - 학습만으로 모든 상황을 막을 수 없음 -> 하네스와 루프 설계로 보완


## 6. 정리
- 모델 안을 바꾸는 방법(1, 2장) / 모델 바깥을 바꾸는 방법(3~5장)
- 상황 -> 무엇을 손볼까
    - 도메인 특화 성능이 필요하고 데이터가 충분하다 -> Fine-tuning / PEFT
    - 모바일에 올려야 한다 -> 경량화
    - 사람이 원하는 답을 못 낸다 -> 정렬(RLHF)
    - API만 쓸 수 있고 출력 형식이 문제다 -> 프롬프트
    - 도구를 쓰거나 파일을 읽어야 한다 -> 하네스
    - 사람이 매번 시키는 게 문제다 -> 루프
- 서로 배타적이지 않아서 여러 개를 함께 쓸 수 있음